In [16]:
import random
class FakeLLM:
    def __init__(self):
        print('LLM initialized')


    def predict(self, prompt):
        
        response_list = [
            'delhi is the capital of india.',
            'paris is the capital of france.',
        ]
        return {'response':random.choice(response_list)}  # Return the first response

class FakePromptTemplate:
    def __init__(self, template, input_variables):
        print('PromptTemplate initialized')
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):
        return self.template.format(**input_dict)

In [17]:
template = FakePromptTemplate(
    template="write a {length} poem about {country}?",
    input_variables=["length","country"]
)   

prompt = template.format({"country": "India", "length": 'short'})
print(prompt)

PromptTemplate initialized
write a short poem about India?


In [22]:
llm = FakeLLM()
response = llm.predict(prompt)
print(response)

LLM initialized
{'response': 'delhi is the capital of india.'}


In [27]:
class FakeLLMChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def run(self, input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)

        return result['response']

In [28]:
template = FakePromptTemplate(
    template="write a {length} poem about {country}?",
    input_variables=["length","country"]
)

PromptTemplate initialized


In [29]:
llm = FakeLLM()

chain = FakeLLMChain(llm, template)

LLM initialized


In [34]:
chain.run({'length':'short','country':'india'})

'paris is the capital of france.'

## Runnable 

In [35]:
from abc import ABC, abstractmethod

In [36]:
class Runnable(ABC):

    @abstractmethod
    def invoke(input_data):
        pass

In [75]:
import random
class FakeLLM(Runnable):
    def __init__(self):
        print('LLM initialized')

    def invoke(self, prompt):   ##Invoke
        
        response_list = [
            'delhi is the capital of india.',
            'paris is the capital of france.',
        ]
        return {'response':random.choice(response_list)}  # Return the first response

    def predict(self, prompt):
        
        response_list = [
            'delhi is the capital of india.',
            'paris is the capital of france.',
        ]
        return {'response':random.choice(response_list)}  # Return the first response

class FakePromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        # print('PromptTemplate initialized')
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):   ##Invoke
        return self.template.format(**input_dict)

    def format(self, input_dict):
        return self.template.format(**input_dict)

In [76]:
class FakeStrOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data['response']

In [77]:
class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [78]:
template = FakePromptTemplate(
    template="write a {length} poem about {country}?",
    input_variables=["length","country"]
)

In [79]:
llm = FakeLLM()

LLM initialized


In [80]:
parser = FakeStrOutputParser()

In [81]:
chain = RunnableConnector([template, llm, parser])

In [82]:
chain.invoke({'length':'short','country':'india'})

'paris is the capital of france.'

## Connecting various Runnables

In [83]:
template1 = FakePromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

template2 = FakePromptTemplate(
    template='Explain the following joke {response}',
    input_variables=['response']
)

In [84]:
llm = FakeLLM()

LLM initialized


In [85]:
parser = FakeStrOutputParser()

chain1 = RunnableConnector([template1,llm])

chain1.invoke({'topic':'AI'})

{'response': 'delhi is the capital of india.'}

In [87]:
chain2 = RunnableConnector([template2,llm, parser])

final_chain = RunnableConnector([chain1, chain2])

final_chain.invoke({'topic':'cricket'})

'paris is the capital of france.'